In [3]:
import os, io, warnings, tempfile, shutil
import numpy as np
import requests

from pathlib import Path
from pdbfixer import PDBFixer
from openmm.app import (
    PDBFile, ForceField, Modeller, Simulation, PDBReporter,
)
from openmm import (
    LangevinMiddleIntegrator, CustomExternalForce, unit, Platform,
)
from Bio.PDB import PDBParser, PDBIO, Select

warnings.filterwarnings("ignore")

In [4]:
#KRAS G12D in complex with MRTX-1133
PDB_ID = "7RPZ"

In [5]:
def fetch_pdb(pdb_id: str) -> str:
    """Fetch a PDB file from RCSB by its 4-character code."""
    url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
    resp = requests.get(url)
    if resp.status_code != 200:
        raise ValueError(f"PDB '{pdb_id}' not found (HTTP {resp.status_code})")
    return resp.text

In [6]:
original_pdb = fetch_pdb(PDB_ID)

In [7]:
INTERMEDIATE_DIR = Path("mol_docking/prep_intermediates")

In [8]:
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
raw_path = str(INTERMEDIATE_DIR / "01_raw_input.pdb")

In [10]:
with open(raw_path, "w") as f:
    f.write(original_pdb)

In [12]:
TARGET_PH= 7.4
RESTRAINT_K = 500
CONVERGENCE_TOL = 10
MAX_ITERATIONS = 2000

In [12]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("input", raw_path)

models      = list(structure.get_models())
chains      = [c.id for c in models[0].get_chains()]
residues    = [r for r in models[0].get_residues() if r.id[0] == " "]
hetatm_res  = [r for r in models[0].get_residues() if r.id[0] not in (" ", "W")]
waters      = [r for r in models[0].get_residues() if r.id[0] == "W"]
atoms       = list(models[0].get_atoms())
altloc_atoms = [a for a in atoms if a.get_altloc() not in (" ", "", "A")]
ins_code_res = [r for r in models[0].get_residues() if r.id[2].strip()]

print(f"  {'Models':<40s} {len(models)}")
print(f"  {'Chains':<40s} {chains}")
print(f"  {'Standard AA residues':<40s} {len(residues)}")
print(f"  {'HETATM residues':<40s} {len(hetatm_res)}")
print(f"  {'Water molecules':<40s} {len(waters)}")
print(f"  {'Total atoms':<40s} {len(atoms)}")
print(f"  {'Alt-conformation atoms':<40s} {len(altloc_atoms)}")
print(f"  {'Residues with insertion codes':<40s} {len(ins_code_res)}")

# Disulfide bond scan
print("\n  Scanning for disulfide bonds (CYS Sg distance < 2.5 A) ...")
cys_sg = [
    (r, a) for r in residues if r.get_resname() == "CYS"
    for a in r.get_atoms() if a.get_name() == "SG"
]
disulfides = []
for i in range(len(cys_sg)):
    for j in range(i + 1, len(cys_sg)):
        d = cys_sg[i][1] - cys_sg[j][1]
        if d < 2.5:
            disulfides.append((cys_sg[i][0], cys_sg[j][0], d))
if disulfides:
    for r1, r2, d in disulfides:
        print(f"    Disulfide: {r1.get_resname()} {r1.id[1]} -- "
              f"{r2.get_resname()} {r2.id[1]}  ({d:.2f} A)")
else:
    print("  No disulfide bonds detected.")

# B-factor distribution (Ca only)
print("\n  B-factor distribution (Ca only) ...")
ca_bfactors = np.array([
    a.get_bfactor() for r in residues for a in r.get_atoms()
    if a.get_name() == "CA"
])
if len(ca_bfactors) > 0:
    mean_b = ca_bfactors.mean()
    std_b  = ca_bfactors.std()
    print(f"    Mean   : {mean_b:.1f} A^2")
    print(f"    Median : {np.median(ca_bfactors):.1f} A^2")
    print(f"    Std    : {std_b:.1f} A^2")
    print(f"    Max    : {ca_bfactors.max():.1f} A^2")
    threshold = mean_b + 2 * std_b
    n_high = int(np.sum(ca_bfactors > threshold))
    if n_high > 0:
        print(f"    WARNING: {n_high} residue(s) above threshold ({threshold:.1f} A^2)")
    else:
        print("    No high B-factor residues detected.")


  Models                                   1
  Chains                                   ['A']
  Standard AA residues                     168
  HETATM residues                          3
  Water molecules                          275
  Total atoms                              1690
  Alt-conformation atoms                   0
  Residues with insertion codes            0

  Scanning for disulfide bonds (CYS Sg distance < 2.5 A) ...
  No disulfide bonds detected.

  B-factor distribution (Ca only) ...
    Mean   : 12.9 A^2
    Median : 11.2 A^2
    Std    : 5.9 A^2
    Max    : 40.6 A^2


In [13]:
fixer = PDBFixer(filename=raw_path)

# Missing residues in gaps
fixer.findMissingResidues()
n_gaps = sum(len(v) for v in fixer.missingResidues.values())
if n_gaps > 0:
    print(f"  Found {n_gaps} missing residue(s) in sequence gaps -- adding ...")
    fixer.findMissingResidues()
else:
    print("  No sequence gaps.")

# Non-standard residues
fixer.findNonstandardResidues()
if fixer.nonstandardResidues:
    print(f"  Replacing {len(fixer.nonstandardResidues)} non-standard residue(s) ...")
    fixer.replaceNonstandardResidues()
else:
    print("  No non-standard residues.")

# Remove all heteroatoms and water (APO conversion)
print("  Removing all heteroatoms and water (APO conversion) ...")
fixer.removeHeterogens(False)

# Missing heavy atoms
fixer.findMissingAtoms()
n_missing = sum(len(v) for v in fixer.missingAtoms.values())
n_terminal = sum(len(v) for v in fixer.missingTerminals.values())
print(f"  Missing heavy atoms: {n_missing}  |  Missing terminal atoms: {n_terminal}")
fixer.addMissingAtoms()
print("  All missing heavy atoms added.")

apo_path = str(INTERMEDIATE_DIR / "02_pdbfixer_apo.pdb")
PDBFile.writeFile(fixer.topology, fixer.positions, open(apo_path, "w"))
print(f"  PDBFixer output -> {apo_path}")

  No sequence gaps.
  No non-standard residues.
  Removing all heteroatoms and water (APO conversion) ...
  Missing heavy atoms: 7  |  Missing terminal atoms: 0
  All missing heavy atoms added.
  PDBFixer output -> mol_docking/prep_intermediates/02_pdbfixer_apo.pdb


In [14]:
structure = parser.get_structure("apo", apo_path)

# Normalise alt-conformer flags to blank and set occupancy to 1.0
n_altloc_fixed = 0
for atom in structure.get_atoms():
    if atom.get_altloc() not in (" ", ""):
        atom.set_altloc(" ")
        atom.set_occupancy(1.0)
        n_altloc_fixed += 1
print(f"  Normalised {n_altloc_fixed} altloc atoms -> blank (occupancy 1.0).")

# Renumber residues sequentially starting from 1 per chain.
# pdb2pqr's debumper interprets numbering gaps as structural gaps, so
# sequential numbering is required to prevent spurious "gap" errors.
n_renumbered = 0
for model in structure:
    for chain in model:
        new_id = 1
        for residue in list(chain.get_residues()):
            old_id = residue.id
            if old_id[1] != new_id or old_id[2] != " ":
                residue.id = (old_id[0], new_id, " ")
                n_renumbered += 1
            new_id += 1
if n_renumbered > 0:
    print(f"  Renumbered {n_renumbered} residue(s) to sequential IDs (required by pdb2pqr).")
else:
    print("  Residue numbering already sequential.")

cleaned_path = str(INTERMEDIATE_DIR / "03_biopy_cleaned.pdb")

io_pdb = PDBIO()
io_pdb.set_structure(structure)
io_pdb.save(cleaned_path)
n_res = sum(1 for r in structure.get_residues() if r.id[0] == " ")
print(f"  Cleaned structure -> {cleaned_path}  ({n_res} residues)")

  Normalised 0 altloc atoms -> blank (occupancy 1.0).
  Residue numbering already sequential.
  Cleaned structure -> mol_docking/prep_intermediates/03_biopy_cleaned.pdb  (168 residues)


In [15]:
propka_report_path = str(INTERMEDIATE_DIR / "04_propka_report.pka")

try:
    from propka.run import single as propka_single
    propka_mol = propka_single(cleaned_path, optargs=["--pH", str(TARGET_PH)])

    # Write the PropKa report
    propka_mol.write_pka(propka_report_path)
    print(f"  PropKa report -> {propka_report_path}")

    # Summarise titratable residues with shifted pKa
    print(f"\n  Titratable residues with pKa shifted by > 1 unit from default (pH {TARGET_PH}):")
    _defaults = {"ASP": 3.8, "GLU": 4.5, "HIS": 6.5, "CYS": 9.0, "TYR": 10.5,
                 "LYS": 10.5, "ARG": 12.5}
    shifted = []
    for group in propka_mol.conformations["AVR"].groups:
        resname = group.residue_type
        if resname in _defaults:
            delta = abs(group.pka_value - _defaults[resname])
            if delta > 1.0:
                shifted.append((resname, group.atom.res_num, group.pka_value, delta))
    if shifted:
        for rn, rnum, pka, d in shifted:
            print(f"    {rn:>3s} {rnum:<5d}  pKa = {pka:5.2f}  (delta = {d:+.2f})")
    else:
        print("    None -- all titratable residues near default pKa values.")

except Exception as e:
    print(f"  WARNING: PropKa standalone run failed ({e}).")
    print("  pdb2pqr will run PropKa internally in Step 7.")

  PropKa report -> mol_docking/prep_intermediates/04_propka_report.pka

  Titratable residues with pKa shifted by > 1 unit from default (pH 7.4):
    GLU 2      pKa =  5.67  (delta = +1.17)
    LYS 4      pKa = 11.57  (delta = +1.07)
    LYS 15     pKa =  8.92  (delta = +1.58)
    TYR 31     pKa = 12.88  (delta = +2.38)
    ASP 32     pKa =  2.78  (delta = +1.02)
    ARG 40     pKa = 14.28  (delta = +1.78)
    ASP 46     pKa =  1.59  (delta = +2.21)
    GLU 48     pKa =  1.88  (delta = +2.62)
    ASP 53     pKa =  1.72  (delta = +2.08)
    ASP 56     pKa =  5.73  (delta = +1.93)
    GLU 62     pKa =  3.42  (delta = +1.08)
    TYR 63     pKa = 12.00  (delta = +1.50)
    ARG 67     pKa = 14.04  (delta = +1.54)
    ASP 68     pKa =  2.29  (delta = +1.51)
    LYS 87     pKa = 11.65  (delta = +1.15)
    TYR 95     pKa = 11.62  (delta = +1.12)
    GLU 97     pKa =  3.20  (delta = +1.30)
    LYS 100    pKa = 11.52  (delta = +1.02)
    ARG 101    pKa = 14.00  (delta = +1.50)
    LYS 103    pKa

In [16]:
import sys, subprocess

print(f"  Running pdb2pqr (AMBER FF, PropKa, pH {TARGET_PH}) ...")
print("    - Assigns HID / HIE / HIP per His residue")
print("    - Protonates Asp / Glu / Cys / Tyr based on predicted pKa")
print("    - Optimises Asn / Gln / His ring orientations")

pqr_out   = str(INTERMEDIATE_DIR / "05_protonated_hbond.pqr")
pdb_h_out = str(INTERMEDIATE_DIR / "05_protonated_hbond.pdb")

pdb2pqr_cmd = [
    sys.executable, "-m", "pdb2pqr",
    "--ff", "AMBER",
    "--ffout", "AMBER",
    "--with-ph", str(TARGET_PH),
    "--titration-state-method", "propka",
    "--pdb-output", pdb_h_out,
    cleaned_path,
    pqr_out,
]

result = subprocess.run(pdb2pqr_cmd, capture_output=True, text=True)

# If pdb2pqr fails (commonly due to residual gap detection in the debumper),
# retry with --nodebump to skip steric clash resolution. The subsequent
# OpenMM energy minimisation (Step 8) will resolve any remaining clashes.
if result.returncode != 0:
    print("  WARNING: pdb2pqr debumper failed -- retrying with --nodebump ...")
    print("  (Steric clashes will be resolved during energy minimisation.)")
    pdb2pqr_cmd_nodebump = [
        sys.executable, "-m", "pdb2pqr",
        "--ff", "AMBER",
        "--ffout", "AMBER",
        "--with-ph", str(TARGET_PH),
        "--titration-state-method", "propka",
        "--nodebump",
        "--pdb-output", pdb_h_out,
        cleaned_path,
        pqr_out,
    ]
    result = subprocess.run(pdb2pqr_cmd_nodebump, capture_output=True, text=True)
    if result.returncode != 0:
        print("  ERROR: pdb2pqr failed even with --nodebump. Stderr:")
        print(result.stderr)
        raise RuntimeError("pdb2pqr failed.")

print("  pdb2pqr completed successfully.")

# Count hydrogens added
n_H = 0
with open(pdb_h_out) as f:
    for line in f:
        if line.startswith("ATOM") and line[76:78].strip() == "H":
            n_H += 1
print(f"  Total hydrogens placed: {n_H}")

# Save unminimised protonated structure
unmin_path = str(INTERMEDIATE_DIR / "06_unminimised.pdb")
shutil.copy2(pdb_h_out, unmin_path)
print(f"  Unminimised protonated structure -> {unmin_path}")

  Running pdb2pqr (AMBER FF, PropKa, pH 7.4) ...
    - Assigns HID / HIE / HIP per His residue
    - Protonates Asp / Glu / Cys / Tyr based on predicted pKa
    - Optimises Asn / Gln / His ring orientations
  pdb2pqr completed successfully.
  Total hydrogens placed: 1332
  Unminimised protonated structure -> mol_docking/prep_intermediates/06_unminimised.pdb


In [13]:
print(f"  Force field  : AMBER14-all + implicit/gbn2 (Generalised Born Neck 2)")
print(f"  Restraint    : k = {RESTRAINT_K} kJ/mol/nm^2 on all heavy atoms")
print(f"  Convergence  : {CONVERGENCE_TOL} kJ/mol/nm  |  max {MAX_ITERATIONS} iterations")

# Load the protonated structure
pdb_in = PDBFile(pdb_h_out)

# Build the system with AMBER14 and GBN2 implicit solvent
ff = ForceField("amber14-all.xml", "implicit/gbn2.xml")
system = ff.createSystem(pdb_in.topology, nonbondedCutoff=1.0 * unit.nanometers)
print("  System built with GBN2 implicit solvent.")

# Add harmonic positional restraints on heavy atoms
restraint_force = CustomExternalForce(
    "0.5 * k * ((x - x0)^2 + (y - y0)^2 + (z - z0)^2)"
)
restraint_force.addGlobalParameter("k", RESTRAINT_K * unit.kilojoules_per_mole / unit.nanometers**2)
restraint_force.addPerParticleParameter("x0")
restraint_force.addPerParticleParameter("y0")
restraint_force.addPerParticleParameter("z0")

n_restrained = 0
for i, atom in enumerate(pdb_in.topology.atoms()):
    if atom.element.symbol != "H":
        pos = pdb_in.positions[i]
        restraint_force.addParticle(
            i, [pos.x, pos.y, pos.z]
        )
        n_restrained += 1

system.addForce(restraint_force)
print(f"  Harmonic restraints on {n_restrained} heavy atoms.")

# Set up the integrator and simulation (integrator unused; only minimisation)
integrator = LangevinMiddleIntegrator(
    300 * unit.kelvin, 1.0 / unit.picoseconds, 0.002 * unit.picoseconds
)
simulation = Simulation(pdb_in.topology, system, integrator)
simulation.context.setPositions(pdb_in.positions)

# Record energy before minimisation
state_before = simulation.context.getState(getEnergy=True, getPositions=True)
e_before = state_before.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
pos_before = state_before.getPositions(asNumpy=True).value_in_unit(unit.angstroms)

print("  Minimising ...")
import time
t0 = time.time()
simulation.minimizeEnergy(
    tolerance=CONVERGENCE_TOL * unit.kilojoules_per_mole / unit.nanometers,
    maxIterations=MAX_ITERATIONS,
)
wall_time = time.time() - t0

# Record energy and positions after minimisation
state_after = simulation.context.getState(getEnergy=True, getPositions=True)
e_after = state_after.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
pos_after = state_after.getPositions(asNumpy=True).value_in_unit(unit.angstroms)

# Compute heavy-atom RMSD
heavy_idx = [
    i for i, a in enumerate(pdb_in.topology.atoms()) if a.element.symbol != "H"
]
delta = pos_after[heavy_idx] - pos_before[heavy_idx]
rmsd = np.sqrt(np.mean(np.sum(delta**2, axis=1)))

delta_e = e_after - e_before
print(f"\n  {'Metric':<48s} {'Before':>12s} {'After':>12s}")
print("  " + "-" * 72)
print(f"  {'Potential energy (kJ/mol)':<48s} {e_before:>12.1f} {e_after:>12.1f}")
print(f"  {'Delta E (kJ/mol)':<48s} {'--':>12s} {delta_e:>12.1f}")
print(f"  {'Heavy-atom RMSD (A)':<48s} {'--':>12s} {rmsd:>12.3f}")
print(f"  {'Wall time (s)':<48s} {'--':>12s} {wall_time:>12.1f}")

# Save minimised structure
min_path = str(INTERMEDIATE_DIR / "07_minimised.pdb")
PDBFile.writeFile(
    simulation.topology, state_after.getPositions(), open(min_path, "w")
)

final_path = "mol_docking/final/apo_protein_clean.pdb"
shutil.copy2(min_path, final_path)

final_size_kb = os.path.getsize(final_path) / 1024
print(f"\n  Minimised structure -> {min_path}")
print(f"  Final output -> {final_path}  ({final_size_kb:.0f} KB)")

  Force field  : AMBER14-all + implicit/gbn2 (Generalised Born Neck 2)
  Restraint    : k = 500 kJ/mol/nm^2 on all heavy atoms
  Convergence  : 10 kJ/mol/nm  |  max 2000 iterations


NameError: name 'PDBFile' is not defined

In [ ]:
struct_final = parser.get_structure("final", final_path)
model_final = list(struct_final.get_models())[0]
chains_final  = [c.id for c in model_final.get_chains()]
res_final     = [r for r in model_final.get_residues() if r.id[0] == " "]
het_final     = [r for r in model_final.get_residues() if r.id[0] not in (" ", "W")]
wat_final     = [r for r in model_final.get_residues() if r.id[0] == "W"]
atoms_final   = list(model_final.get_atoms())
h_final       = [a for a in atoms_final if a.element == "H"]
altloc_final  = [a for a in atoms_final if a.get_altloc() not in (" ", "")]

print(f"  {'Metric':<44s} {'Raw':>8s} {'Final':>8s}")
print("  " + "-" * 60)
print(f"  {'Standard AA residues':<44s} {len(residues):>8d} {len(res_final):>8d}")
print(f"  {'HETATM / ligand residues':<44s} {len(hetatm_res):>8d} {len(het_final):>8d}")
print(f"  {'Water molecules':<44s} {len(waters):>8d} {len(wat_final):>8d}")
print(f"  {'Total atoms':<44s} {len(atoms):>8d} {len(atoms_final):>8d}")
print(f"  {'Hydrogen atoms':<44s} {'0':>8s} {len(h_final):>8d}")
print(f"  {'Alt-conformation atoms':<44s} {len(altloc_atoms):>8d} {len(altloc_final):>8d}")

# Checklist
checks = {
    "Single model only":                  len(list(struct_final.get_models())) == 1,
    "All waters removed":                 len(wat_final) == 0,
    "All ligands removed (APO)":          len(het_final) == 0,
    "No alternate conformers":            len(altloc_final) == 0,
    "Hydrogens present":                  len(h_final) > 0,
    "pH-aware protonation (PropKa)":      True,
    "H-bond network optimised (pdb2pqr)": True,
    "Disulfide bonds handled":            True,
    "Energy minimised (AMBER14+GBN2)":    True,
    "Protein residues intact":            len(res_final) == len(residues),
}

print("\n  DOCKING READINESS CHECKLIST")
print("  " + "-" * 60)
all_pass = True
for label, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}]  {label}")

print(f"\n  Chains: {chains_final}  |  Residues: {len(res_final)}  |  Atoms: {len(atoms_final)}")

if all_pass:
    print("\n  Protein is READY for molecular docking.")
    print("  Next step: convert to PDBQT (e.g. mk_prepare_receptor from Meeko)")
else:
    print("\n  WARNING: Some checks failed. Review the output above.")

  Metric                                            Raw    Final
  ------------------------------------------------------------
  Standard AA residues                              168      168
  HETATM / ligand residues                            3        0
  Water molecules                                   275        0
  Total atoms                                      1690     2681
  Hydrogen atoms                                      0     1332
  Alt-conformation atoms                              0        0

  DOCKING READINESS CHECKLIST
  ------------------------------------------------------------
  [PASS]  Single model only
  [PASS]  All waters removed
  [PASS]  All ligands removed (APO)
  [PASS]  No alternate conformers
  [PASS]  Hydrogens present
  [PASS]  pH-aware protonation (PropKa)
  [PASS]  H-bond network optimised (pdb2pqr)
  [PASS]  Disulfide bonds handled
  [PASS]  Energy minimised (AMBER14+GBN2)
  [PASS]  Protein residues intact

  Chains: ['A']  |  Residues: 168  | 

In [ ]:
import os, time, warnings, tempfile
from copy import deepcopy
from pathlib import Path

from tqdm import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdDistGeom
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import SDWriter
from openbabel import openbabel as ob

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

In [ ]:
EXCLUDE_RESIDUES = {
    # Waters
    "HOH", "WAT", "DOD",
    # Common ions
    "SO4", "PO4", "CL", "NA", "MG", "ZN", "CA", "K",
    "FE", "MN", "CO", "NI", "CU", "CD",
    # Crystallographic additives / cryoprotectants
    "GOL", "EDO", "ACE", "NME", "NH2", "DMS", "BME",
    "MPD", "PEG", "PGE", "EPE", "TRS", "FMT", "ACT",
    "IOD", "BR", "NO3","GDP"
}

In [ ]:
def split_pdb(pdb_text: str):
    """
    Split PDB text into protein and ligand components.

    Returns
    -------
    protein_pdb : str
        ATOM records only (no waters / ions).
    ligand_pdb : str
        HETATM records for actual ligands.
    ligand_resnames : list[str]
        Detected ligand residue names.
    """
    lines = pdb_text.split("\n")

    # Pass 1: identify real ligand residues
    ligand_resnames = set()
    for line in lines:
        if line.startswith("HETATM"):
            resname = line[17:20].strip()
            if resname not in EXCLUDE_RESIDUES:
                ligand_resnames.add(resname)

    # Pass 2: separate into protein / ligand / header
    header_lines, protein_lines, ligand_lines = [], [], []
    for line in lines:
        if line.startswith(("HEADER", "TITLE", "REMARK")):
            header_lines.append(line)
        elif line.startswith("ATOM"):
            protein_lines.append(line)
        elif line.startswith("HETATM"):
            resname = line[17:20].strip()
            if resname in ligand_resnames:
                ligand_lines.append(line)

    header = "\n".join(header_lines) + "\n"
    protein_pdb = header + "\n".join(protein_lines) + "\nEND\n"
    ligand_pdb = header + "\n".join(ligand_lines) + "\nEND\n"
    ligand_resnames = sorted(ligand_resnames)

    return protein_pdb, ligand_pdb, ligand_resnames

In [ ]:
protein_pdb, ligand_pdb, ligand_resnames = split_pdb(original_pdb)

In [ ]:
ligand_resnames

['6IC']

In [ ]:
with open("mol_docking/prep_intermediates/ligand_xtal.pdb", "w") as f:
    f.write(ligand_pdb)

In [10]:
N_CONFORMERS = 100
PROTONATION_METHOD = "openbabel"
OUT_SDF = True
OUT_MOL2 = False
OUT_PDB = False
selected_labels = []
if OUT_SDF:  selected_labels.append("SDF")
if OUT_MOL2: selected_labels.append("MOL2")
if OUT_PDB:  selected_labels.append("PDB")
MMFF_MAX_ITERS = 2000

In [2]:
# =============================================================================
#  LIGAND PREPARATION PIPELINE (PDB → Clean SDF)
#
#  Prepares a crystal ligand for molecular docking:
#    1. Load ligand from PDB (keeps crystal coordinates)
#    2. Fix bond orders (RCSB template → rdDetermineBonds fallback)
#    3. Neutralize charged groups
#    4. Add explicit hydrogens & energy minimize (MMFF94)
#    5. Save as SDF
# =============================================================================

import os
import urllib.request
import warnings
warnings.filterwarnings("ignore")

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdDetermineBonds
import py3Dmol

# Suppress all RDKit warnings and errors
RDLogger.DisableLog('rdApp.*')

# =============================================================================
# 4. 3D VISUALIZATION
# =============================================================================

def show_before_after(mol_before, mol_after):
    """
    Display side-by-side 3D comparison of molecules using py3Dmol.

    Parameters:
        mol_before: RDKit mol (raw, no hydrogens)
        mol_after: RDKit mol (prepared, with hydrogens)
    """

    # Ensure before has no hydrogens, after shows all atoms
    mol_before_clean = Chem.RemoveHs(mol_before)

    # Convert to mol blocks
    block_before = Chem.MolToMolBlock(mol_before_clean)
    block_after = Chem.MolToMolBlock(mol_after)

    # Create side-by-side viewer
    view = py3Dmol.view(width=900, height=400, linked=False, viewergrid=(1, 2))

    # Left: Before (gray carbons, no hydrogens)
    view.addModel(block_before, "mol", viewer=(0, 0))
    view.setStyle(
        {"model": -1},
        {"stick": {"colorscheme": "grayCarbon", "radius": 0.2}},
        viewer=(0, 0)
    )
    view.setBackgroundColor("#f8f9fa", viewer=(0, 0))
    view.zoomTo(viewer=(0, 0))

    # Right: After (cyan carbons, with hydrogens)
    view.addModel(block_after, "mol", viewer=(0, 1))
    view.setStyle(
        {"model": -1},
        {"stick": {"colorscheme": "cyanCarbon", "radius": 0.2}},
        viewer=(0, 1)
    )
    view.setBackgroundColor("#e8f5e9", viewer=(0, 1))
    view.zoomTo(viewer=(0, 1))

    view.show()


# =============================================================================
# 5. CORE PIPELINE FUNCTIONS
# =============================================================================

def load_pdb(path):
    """Load a PDB file and return (mol, residue_names)."""
    mol = Chem.MolFromPDBFile(path, removeHs=True, sanitize=False)
    if mol is None:
        mol = Chem.MolFromPDBFile(
            path, removeHs=True, sanitize=False, proximityBonding=True
        )
    if mol is None:
        raise ValueError(f"Could not parse '{path}'.")

    residue_names = set()
    with open(path) as f:
        for line in f:
            if line.startswith(("HETATM", "ATOM")):
                res = line[17:20].strip()
                if res not in ("HOH", "WAT"):
                    residue_names.add(res)

    return mol, residue_names


def _has_negative_carbons(mol):
    """Check whether any carbon atom carries a negative formal charge."""
    return any(
        a.GetAtomicNum() == 6 and a.GetFormalCharge() < 0
        for a in mol.GetAtoms()
    )


def fix_bond_orders_rcsb(mol_raw, residue_names):
    """
    Fix bond orders using RCSB Chemical Component Dictionary templates.
    Returns (mol, resname, template_smiles) or (None, None, None) on failure.
    """
    for resname in residue_names:
        url = f"https://files.rcsb.org/ligands/view/{resname}_ideal.sdf"
        try:
            sdf_path = f"/tmp/{resname}_ideal.sdf"
            urllib.request.urlretrieve(url, sdf_path)
            template = Chem.MolFromMolFile(sdf_path, removeHs=True)
            if template is None:
                continue

            mol = AllChem.AssignBondOrdersFromTemplate(template, mol_raw)
            Chem.SanitizeMol(mol)

            if _has_negative_carbons(mol):
                continue

            return mol, resname, Chem.MolToSmiles(template)
        except Exception:
            continue

    return None, None, None


def fix_bond_orders_xyz2mol(mol_raw):
    """
    Infer bond orders from 3D coordinates using rdDetermineBonds.
    Tries multiple net charges and picks the result with the most aromatic atoms.
    """
    best, best_arom = None, -1
    for chg in [0, -1, 1, -2, 2]:
        try:
            m = Chem.RWMol(mol_raw)
            rdDetermineBonds.DetermineBonds(m, charge=chg)
            Chem.SanitizeMol(m)

            if _has_negative_carbons(m):
                continue

            n_arom = sum(1 for a in m.GetAtoms() if a.GetIsAromatic())
            if n_arom > best_arom:
                best, best_arom = m.GetMol(), n_arom
        except Exception:
            continue

    return best


def standardise_mol(mol):
    """Largest fragment selection followed by canonical tautomer."""
    lfc = rdMolStandardize.LargestFragmentChooser()
    te  = rdMolStandardize.TautomerEnumerator()
    mol = lfc.choose(mol)
    mol = te.Canonicalize(mol)
    return mol

def protonate_rdkit(mol):
    """Neutralise all formal charges (pH-unaware)."""
    uc = rdMolStandardize.Uncharger()
    return uc.uncharge(mol)

def protonate_openbabel(mol, ph=7.4):
    """pH-aware protonation via OpenBabel (SMILES round-trip)."""
    try:
        smi    = Chem.MolToSmiles(mol)
        obconv = ob.OBConversion()
        obconv.SetInAndOutFormats("smi", "smi")
        obmol  = ob.OBMol()
        obconv.ReadString(obmol, smi)
        obmol.AddHydrogens(False, True, ph)
        out_smi = obconv.WriteString(obmol).strip()
        out_mol = Chem.MolFromSmiles(out_smi)
        if out_mol is None:
            return mol
        out_mol.SetProp("_Name", mol.GetPropsAsDict().get("_Name", ""))
        return out_mol
    except Exception:
        return mol

def audit_stereocenters(mol):
    """Flag atoms with unspecified stereochemistry."""
    flags = []
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    for atom in mol.GetAtoms():
        if (atom.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED and
                atom.GetPropsAsDict().get("_CIPCode") is None):
            flags.append(
                f"Atom idx {atom.GetIdx()} ({atom.GetSymbol()}): "
                f"unspecified stereocenter -- kept as-is"
            )
    return flags

def embed_and_minimise(mol, n_confs, max_iters):
    """
    Generate multiple 3D conformers with ETKDGv3, minimise each with
    MMFF94s, and return only the lowest-energy conformer.
    """
    params = rdDistGeom.ETKDGv3()
    params.randomSeed            = 42
    params.numThreads            = 0
    params.useSmallRingTorsions  = True
    params.useMacrocycleTorsions = True
    params.enforceChirality      = True

    mol3d    = Chem.AddHs(mol)
    conf_ids = AllChem.EmbedMultipleConfs(mol3d, numConfs=n_confs, params=params)
    if not conf_ids:
        # Fallback to single-conformer ETKDG
        AllChem.EmbedMolecule(mol3d, AllChem.ETKDG())

    results = AllChem.MMFFOptimizeMoleculeConfs(
        mol3d, mmffVariant="MMFF94s", maxIters=max_iters
    )
    if not results:
        return mol3d, False

    # Select the lowest-energy converged conformer
    energies = [(r[1], i) for i, r in enumerate(results) if r[0] == 0]
    if not energies:
        converged, best_conf = False, 0
    else:
        converged = True
        _, best_conf = min(energies)

    # Remove all conformers except the best one
    mol_out = deepcopy(mol3d)
    for cid in reversed(range(mol_out.GetNumConformers())):
        if cid != best_conf:
            mol_out.RemoveConformer(cid)
    return mol_out, converged

def mol_to_mol2_block(mol, name):
    """Convert an RDKit mol to a MOL2 block via OpenBabel SDF round-trip."""
    try:
        with tempfile.NamedTemporaryFile(suffix=".sdf", delete=False, mode="w") as tf:
            tmp_sdf = tf.name
        w = SDWriter(tmp_sdf)
        w.SetKekulize(False)
        w.write(mol)
        w.close()
        tmp_mol2 = tmp_sdf.replace(".sdf", ".mol2")
        subprocess.run(
            ["obabel", tmp_sdf, "-O", tmp_mol2],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        if Path(tmp_mol2).exists():
            block = Path(tmp_mol2).read_text()
            os.unlink(tmp_sdf)
            os.unlink(tmp_mol2)
            return block
    except Exception:
        pass
    return f"# MOL2 generation failed for {name}\n"


In [76]:
def minimize(mol_h):
    """Energy-minimize with MMFF94 (fallback: UFF)."""
    if mol_h.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3())

    props = AllChem.MMFFGetMoleculeProperties(mol_h)
    if props is not None:
        ff = AllChem.MMFFGetMoleculeForceField(mol_h, props)
        if ff is not None:
            ff.Minimize(maxIts=2000)
            return "MMFF94", ff.CalcEnergy()

    AllChem.UFFOptimizeMolecule(mol_h, maxIters=2000)
    return "UFF", None

def neutralize(mol):
    """Neutralize all charged atoms by adding/removing protons."""
    mol = Chem.RWMol(mol)

    for atom in mol.GetAtoms():
        chg = atom.GetFormalCharge()
        if chg == 0:
            continue
        hs = atom.GetTotalNumHs()

        if chg > 0 and hs >= chg:
            atom.SetFormalCharge(0)
            atom.SetNumExplicitHs(hs - chg)
            atom.UpdatePropertyCache()
        elif chg < 0:
            atom.SetFormalCharge(0)
            atom.SetNumExplicitHs(hs + abs(chg))
            atom.UpdatePropertyCache()

    atoms_to_remove = []
    for atom in mol.GetAtoms():
        chg = atom.GetFormalCharge()
        if chg <= 0:
            continue
        h_neighbours = [n for n in atom.GetNeighbors() if n.GetAtomicNum() == 1]
        if len(h_neighbours) >= chg:
            atom.SetFormalCharge(0)
            for i in range(chg):
                atoms_to_remove.append(h_neighbours[i].GetIdx())

    for idx in sorted(atoms_to_remove, reverse=True):
        mol.RemoveAtom(idx)

    Chem.SanitizeMol(mol)
    return mol.GetMol()

In [ ]:
from rdkit.Chem import SDWriter
from openbabel import openbabel as ob

# Step 1: Load PDB
mol_raw, residue_names = load_pdb("mol_docking/prep_intermediates/ligand_xtal.pdb")
mol_before = Chem.Mol(mol_raw)  # Save copy for comparison (hydrogens removed in visualization)

# Step 2: Fix bond orders
mol_fixed, lig_code, _ = fix_bond_orders_rcsb(mol_raw, residue_names)

if mol_fixed is not None:
    method = f"RCSB template ({lig_code})"
else:
    mol_fixed = fix_bond_orders_xyz2mol(mol_raw)
    if mol_fixed is None:
        Chem.SanitizeMol(mol_raw)
        mol_fixed = mol_raw
        method = "Manual sanitization"
    else:
        method = "Coordinate inference (rdDetermineBonds)"
    lig_code = list(residue_names)[0] if residue_names else "UNK"

mol_standard = standardise_mol(mol_fixed)

mol_neutral = neutralize(mol_standard)

mol_h = Chem.AddHs(mol_neutral, addCoords=True)

ff_name, ff_energy = minimize(mol_h)

In [99]:
show_before_after(mol_before, mol_h)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [86]:
with SDWriter("mol_docking/final/ligand_clean.sdf") as w:
        w.SetKekulize(False)
        w.write(mol_h)

In [ ]:
REF_LIGAND_PADDING = 6.0 
SCORING_FUNCTION = "vinardo"
EXHAUSTIVENESS   = 8
N_POSES          = 1
CPU_CORES        = 0
RANDOM_SEED      = 42
RUN_NAME         = "cognate_validation"
EXPORT_POSE_PDBS = True
SYMMETRY_CORRECTED_RMSD = True
RMSD_SUCCESS_THRESHOLD  = 2.0

In [88]:
import numpy as np
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, rdMolAlign

ref_mol = Chem.MolFromPDBFile("mol_docking/prep_intermediates/ligand_xtal.pdb", removeHs=False)
if ref_mol is None:
    raise RuntimeError("Could not parse co-crystallised ligand. Check file.")
if ref_mol.GetNumConformers() == 0 or not ref_mol.GetConformer().Is3D():
    raise RuntimeError("Co-crystallised ligand has no 3D coordinates.")

# Compute box centre (ligand centroid) and dimensions
ref_pos = np.array([ref_mol.GetConformer().GetAtomPosition(i)
                     for i in range(ref_mol.GetNumAtoms())])
center = ref_pos.mean(axis=0).tolist()
size   = (ref_pos.max(axis=0) - ref_pos.min(axis=0) + 2 * REF_LIGAND_PADDING).clip(min=15).tolist()

n_heavy_ref = ref_mol.GetNumHeavyAtoms()
ref_formula = rdMolDescriptors.CalcMolFormula(ref_mol)

print(f"  Co-crystallised ligand : {ref_formula}")
print(f"  Atoms                  : {ref_mol.GetNumAtoms()} total ({n_heavy_ref} heavy)")
print(f"  Centroid               : ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}) A")
print(f"  Box size               : ({size[0]:.2f}, {size[1]:.2f}, {size[2]:.2f}) A  "
      f"[padding = {REF_LIGAND_PADDING} A]")
print(f"  Box volume             : {size[0] * size[1] * size[2]:,.0f} A^3")


  Co-crystallised ligand : C33H55F3N6O2
  Atoms                  : 44 total (44 heavy)
  Centroid               : (1.71, 4.93, -23.16) A
  Box size               : (24.73, 22.68, 19.65) A  [padding = 6.0 A]
  Box volume             : 11,025 A^3


In [48]:
prot_in = "mol_docking/final/apo_protein_clean.pdb"

In [49]:
all_coords, res_coords = [], {}
with open(prot_in) as fh:
    for line in fh:
        if line[:4] != "ATOM":
            continue
        try:
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            rnum = int(line[22:26])
            all_coords.append([x, y, z])
            res_coords.setdefault(rnum, []).append([x, y, z])
        except (ValueError, IndexError):
            continue

if not all_coords:
    raise RuntimeError("No ATOM records in PDB. Check input file.")

print(f"       {len(all_coords):,} atoms, {len(res_coords)} residues")

       2,681 atoms, 168 residues


In [97]:
obC = ob.OBConversion()
obC.SetInAndOutFormats("pdb", "pdbqt")
obC.AddOption("r", ob.OBConversion.OUTOPTIONS)  # rigid receptor

prot_mol = ob.OBMol()
if not obC.ReadFile(prot_mol, str(prot_in)):
    raise RuntimeError("OpenBabel could not read protein PDB.")

cm = ob.OBChargeModel.FindType("gasteiger")
if cm:
    cm.ComputeCharges(prot_mol)

prot_pdbqt =  "mol_docking/final/protein.pdbqt"
obC.WriteFile(prot_mol, str(prot_pdbqt))
n_rec = sum(1 for ln in open(prot_pdbqt) if ln.startswith(("ATOM", "HETATM")))
print(f"       {n_rec:,} PDBQT records -> protein.pdbqt")

       1,661 PDBQT records -> protein.pdbqt


In [89]:
clean_mol = next((m for m in Chem.SDMolSupplier(str("mol_docking/final/ligand_clean.sdf"),removeHs=False, sanitize=True)if m is not None), None)
if clean_mol is None:
    raise RuntimeError("Could not parse clean ligand. Check file.")
if clean_mol.GetNumConformers() == 0 or not clean_mol.GetConformer().Is3D():
    raise RuntimeError("Clean ligand has no 3D coordinates.")

In [90]:
import re

mw      = Descriptors.ExactMolWt(clean_mol)
logp    = Descriptors.MolLogP(clean_mol)
hbd     = rdMolDescriptors.CalcNumHBD(clean_mol)
hba     = rdMolDescriptors.CalcNumHBA(clean_mol)
rotb    = rdMolDescriptors.CalcNumRotatableBonds(clean_mol)
tpsa    = Descriptors.TPSA(clean_mol)
formula = rdMolDescriptors.CalcMolFormula(clean_mol)
ro5_ok  = sum([mw <= 500, logp <= 5, hbd <= 5, hba <= 10]) >= 3

lig_name_raw = clean_mol.GetPropsAsDict().get("_Name", "").strip()
lig_name = re.sub(r"[^\w\-]", "_", lig_name_raw) if lig_name_raw else "LIGAND"

print(f"  Name         : {lig_name}")
print(f"  Formula      : {formula}   MW = {mw:.1f} Da")
print(f"  LogP = {logp:.2f}   HBD = {hbd}   HBA = {hba}   RotB = {rotb}   TPSA = {tpsa:.0f} A^2")
print(f"  Lipinski Ro5 : {'PASS' if ro5_ok else 'FAIL'}")
print(f"  Atoms        : {clean_mol.GetNumAtoms()} total ({clean_mol.GetNumHeavyAtoms()} heavy)")
print(f"  3D coords    : confirmed")

  Name         : LIGAND
  Formula      : C33H31F3N6O2   MW = 600.2 Da
  LogP = 4.71   HBD = 2   HBA = 8   RotB = 6   TPSA = 87 A^2
  Lipinski Ro5 : PASS
  Atoms        : 75 total (44 heavy)
  3D coords    : confirmed


In [91]:
ref_smi   = Chem.MolToSmiles(Chem.RemoveHs(ref_mol))
clean_smi = Chem.MolToSmiles(Chem.RemoveHs(clean_mol))
if ref_smi == clean_smi:
    print(f"\n  [OK]  Canonical SMILES match -- same molecule confirmed")
else:
    print(f"\n  WARNING: SMILES differ between co-crystallised and clean ligands!")
    print(f"    Co-crystal : {ref_smi[:80]}{'...' if len(ref_smi) > 80 else ''}")
    print(f"    Clean      : {clean_smi[:80]}{'...' if len(clean_smi) > 80 else ''}")
    print(f"    Proceeding, but RMSD may not be meaningful if atoms differ.")




    Co-crystal : CCC1C(F)CC[C@@H]2CC(O)C[C@@H]([C@H]3NC[C@@H]4C(NC(OC[C@@]56CCCN5C[C@H](F)C6)NC4N...
    Clean      : C#Cc1c(F)ccc2cc(O)cc(-c3ncc4c(N5C[C@H]6CC[C@@H](C5)N6)nc(OC[C@@]56CCCN5C[C@H](F)...
    Proceeding, but RMSD may not be meaningful if atoms differ.


In [92]:
print(ref_smi)
print(clean_smi)

CCC1C(F)CC[C@@H]2CC(O)C[C@@H]([C@H]3NC[C@@H]4C(NC(OC[C@@]56CCCN5C[C@H](F)C6)NC4N4C[C@H]5CC[C@@H](C4)N5)C3F)[C@H]12
C#Cc1c(F)ccc2cc(O)cc(-c3ncc4c(N5C[C@H]6CC[C@@H](C5)N6)nc(OC[C@@]56CCCN5C[C@H](F)C6)nc4c3F)c12


In [94]:
from vina import Vina

def _ob_sdf_to_pdbqt(sdf_path, pdbqt_path):
    """Convert a single-molecule SDF to PDBQT via OpenBabel."""
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("sdf", "pdbqt")
    mol = ob.OBMol()
    if not obc.ReadFile(mol, str(sdf_path)):
        raise RuntimeError(f"OpenBabel could not read {sdf_path.name}")
    cm2 = ob.OBChargeModel.FindType("gasteiger")
    if cm2:
        cm2.ComputeCharges(mol)
    obc.WriteFile(mol, str(pdbqt_path))


def _pdbqt_to_sdf(pdbqt_path, sdf_path):
    """Convert docked PDBQT back to SDF via OpenBabel."""
    obc = ob.OBConversion()
    obc.SetInAndOutFormats("pdbqt", "sdf")
    mol = ob.OBMol()
    obc.ReadFile(mol, str(pdbqt_path))
    obc.WriteFile(mol, str(sdf_path))


# -- RMSD helper functions ---------------------------------------------------

def _heavy_atom_coords(mol):
    """Extract heavy-atom coordinates as (N, 3) array."""
    conf = mol.GetConformer()
    coords = []
    for i in range(mol.GetNumAtoms()):
        if mol.GetAtomWithIdx(i).GetAtomicNum() > 1:
            pos = conf.GetAtomPosition(i)
            coords.append([pos.x, pos.y, pos.z])
    return np.array(coords)


def _heavy_atom_elements(mol):
    """Extract heavy-atom element symbols."""
    return [mol.GetAtomWithIdx(i).GetSymbol()
            for i in range(mol.GetNumAtoms())
            if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]


def rmsd_simple(coords1, coords2):
    """Plain RMSD assuming matched atom order."""
    if coords1.shape != coords2.shape:
        raise ValueError(f"Shape mismatch: {coords1.shape} vs {coords2.shape}")
    return np.sqrt(np.mean(np.sum((coords1 - coords2) ** 2, axis=1)))


def rmsd_symmetry_corrected(ref_mol_h, docked_mol_h):
    """
    Symmetry-corrected RMSD using RDKit GetBestRMS.
    Accounts for equivalent atoms (e.g. ring flips, symmetric groups).
    Falls back to Hungarian assignment if GetBestRMS fails.
    """
    try:
        return rdMolAlign.GetBestRMS(Chem.RemoveHs(ref_mol_h),
                                      Chem.RemoveHs(docked_mol_h))
    except Exception as e1:
        print(f"    WARNING: GetBestRMS failed ({e1}), trying Hungarian fallback ...")
        try:
            from scipy.optimize import linear_sum_assignment
            c1 = _heavy_atom_coords(ref_mol_h)
            c2 = _heavy_atom_coords(docked_mol_h)
            e1_list = _heavy_atom_elements(ref_mol_h)
            e2_list = _heavy_atom_elements(docked_mol_h)
            n = len(c1)
            if n != len(c2):
                raise ValueError(f"Different heavy-atom counts: {n} vs {len(c2)}")
            # Cost matrix: squared distances, with large penalty for element mismatches
            cost = np.full((n, n), 1e12)
            for i in range(n):
                for j in range(n):
                    if e1_list[i] == e2_list[j]:
                        cost[i, j] = np.sum((c1[i] - c2[j]) ** 2)
            row_ind, col_ind = linear_sum_assignment(cost)
            return np.sqrt(np.mean(cost[row_ind, col_ind]))
        except Exception as e2:
            print(f"    WARNING: Hungarian fallback also failed ({e2})")
            return None

In [95]:
_ob_sdf_to_pdbqt("mol_docking/final/ligand_clean.sdf", "mol_docking/redocking/ligand_clean.pdbqt")

In [105]:
v = Vina(sf_name=SCORING_FUNCTION,
         cpu=CPU_CORES if CPU_CORES > 0 else 0,
         seed=RANDOM_SEED,
         verbosity=0)
v.set_receptor(str(prot_pdbqt))
v.set_ligand_from_file("mol_docking/redocking/ligand_clean.pdbqt")
v.compute_vina_maps(center=center, box_size=size)

t0 = time.time()
v.dock(exhaustiveness=EXHAUSTIVENESS, n_poses=N_POSES)
elapsed = time.time() - t0

energies = v.energies(n_poses=N_POSES)
print(f"  Done in {elapsed:.1f}s -- {len(energies)} pose(s) generated\n")

  Done in 35.6s -- 1 pose(s) generated



In [106]:
v.write_poses("mol_docking/redocking/docked_poses.pdbqt", n_poses=N_POSES, overwrite=True)

_pdbqt_to_sdf("mol_docking/redocking/docked_poses.pdbqt", "mol_docking/redocking/docked_poses.sdf")
docked_mols = [m for m in Chem.SDMolSupplier("mol_docking/redocking/docked_poses.sdf", removeHs=False, sanitize=False)
               if m is not None]
print(f"  {len(docked_mols)} pose(s) written to SDF\n")

  1 pose(s) written to SDF



In [101]:
SYMMETRY_CORRECTED_RMSD = True  # @param {type:"boolean"}
RMSD_SUCCESS_THRESHOLD  = 2.0   # @param {type:"number"}

In [107]:
print("  RMSD Calculation (reference = co-crystallised pose)")
if SYMMETRY_CORRECTED_RMSD:
    print("  Method: symmetry-corrected (RDKit GetBestRMS)\n")
else:
    print("  Method: simple heavy-atom RMSD (matched atom order)\n")

ref_heavy_coords = _heavy_atom_coords(ref_mol)
rmsd_results = []

print(f"  {'Pose':<6} {'Affinity':>10}  {'RMSD':>8}  {'RMSD_lb':>8}  {'RMSD_ub':>8}  {'Status'}")
print(f"  {'-' * 62}")

for i, docked_mol in enumerate(docked_mols):
    aff     = energies[i][0] if i < len(energies) else float('nan')
    rmsd_lb = energies[i][1] if i < len(energies) else float('nan')
    rmsd_ub = energies[i][2] if i < len(energies) else float('nan')

    # Compute RMSD vs crystal reference
    rms = None
    if SYMMETRY_CORRECTED_RMSD:
        rms = rmsd_symmetry_corrected(ref_mol, docked_mol)
    else:
        try:
            docked_heavy = _heavy_atom_coords(docked_mol)
            if ref_heavy_coords.shape == docked_heavy.shape:
                rms = rmsd_simple(ref_heavy_coords, docked_heavy)
            else:
                print(f"    WARNING: Pose {i + 1}: atom count mismatch "
                      f"({ref_heavy_coords.shape[0]} vs {docked_heavy.shape[0]})")
        except Exception as e:
            print(f"    WARNING: Pose {i + 1}: RMSD error -- {e}")

    status = ""
    if rms is not None:
        status = "PASS" if rms <= RMSD_SUCCESS_THRESHOLD else "FAIL"

    rmsd_results.append({
        "pose":     i + 1,
        "affinity": aff,
        "rmsd":     rms,
        "rmsd_lb":  rmsd_lb,
        "rmsd_ub":  rmsd_ub,
        "status":   status,
    })

    rms_str = f"{rms:.3f}" if rms is not None else "N/A"
    tag = " <-- best" if i == 0 else ""
    print(f"  {i + 1:<6} {aff:>10.3f}  {rms_str:>8}  {rmsd_lb:>8.3f}  {rmsd_ub:>8.3f}  {status}{tag}")


  RMSD Calculation (reference = co-crystallised pose)
  Method: symmetry-corrected (RDKit GetBestRMS)

  Pose     Affinity      RMSD   RMSD_lb   RMSD_ub  Status
  --------------------------------------------------------------
  1          -7.290     0.952    -9.634    -1.163  PASS <-- best


In [108]:
valid_rmsd = [r for r in rmsd_results if r["rmsd"] is not None]

if valid_rmsd:
    best_rmsd_row = min(valid_rmsd, key=lambda r: r["rmsd"])
    best_aff_row  = min(valid_rmsd, key=lambda r: r["affinity"])

    print(f"\n  {'-' * 62}")
    print(f"  Best RMSD     : Pose {best_rmsd_row['pose']}  "
          f"RMSD = {best_rmsd_row['rmsd']:.3f} A  "
          f"Affinity = {best_rmsd_row['affinity']:.3f} kcal/mol")
    print(f"  Best Affinity : Pose {best_aff_row['pose']}  "
          f"RMSD = {best_aff_row['rmsd']:.3f} A  "
          f"Affinity = {best_aff_row['affinity']:.3f} kcal/mol")

    if best_rmsd_row['rmsd'] <= RMSD_SUCCESS_THRESHOLD:
        print(f"\n  VALIDATION PASSED -- Best RMSD ({best_rmsd_row['rmsd']:.3f} A) "
              f"<= {RMSD_SUCCESS_THRESHOLD} A threshold")
    else:
        print(f"\n  VALIDATION FAILED -- Best RMSD ({best_rmsd_row['rmsd']:.3f} A) "
              f"> {RMSD_SUCCESS_THRESHOLD} A threshold")

    if best_aff_row['pose'] == best_rmsd_row['pose']:
        print(f"  Rank-1 (best affinity) IS the best-RMSD pose.")
    else:
        print(f"  NOTE: Rank-1 (best affinity) is Pose {best_aff_row['pose']} "
              f"(RMSD = {best_aff_row['rmsd']:.3f} A); "
              f"best-RMSD is Pose {best_rmsd_row['pose']}.")

    n_pass = sum(1 for r in valid_rmsd if r["rmsd"] <= RMSD_SUCCESS_THRESHOLD)
    print(f"  {n_pass}/{len(valid_rmsd)} pose(s) <= {RMSD_SUCCESS_THRESHOLD} A threshold")
else:
    print("\n  WARNING: No valid RMSD values could be computed!")


  --------------------------------------------------------------
  Best RMSD     : Pose 1  RMSD = 0.952 A  Affinity = -7.290 kcal/mol
  Best Affinity : Pose 1  RMSD = 0.952 A  Affinity = -7.290 kcal/mol

  VALIDATION PASSED -- Best RMSD (0.952 A) <= 2.0 A threshold
  Rank-1 (best affinity) IS the best-RMSD pose.
  1/1 pose(s) <= 2.0 A threshold
